<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Solutions</h2>
<h2>Notebook B03: Advanced Statistical Methods</h2>
</div>

Worked solutions to the 3 exercises in
[Notebook B03: Advanced Statistical Methods](../notebooks/B03_Advanced_statistical_models.ipynb).

**Try each exercise yourself first.** These notebooks are most useful as a check on your reasoning, and
least useful as something to read straight through. An exercise you attempted and got wrong teaches more
than a solution you agreed with.

Where an exercise asks a question rather than requesting code, the answer is written out under the code
that produces it. Several of them have answers that are more interesting than they look.

The setup cell below reproduces the state the exercises assume, so this notebook runs on its own.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="setup">Setup</h3>
</div>

Exercise 3 needs Prophet, which is optional: `uv sync --group advanced`. The first two run on the default
install.

In [ ]:
import importlib.util
import logging
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.deterministic import DeterministicProcess, Fourier
from statsmodels.tsa.forecasting.theta import ThetaModel
from statsmodels.tsa.statespace.sarimax import SARIMAX

sys.path.append("../notebooks")
import nb_config

sns.set_theme(style="whitegrid")
warnings.filterwarnings("ignore")

PROPHET_AVAILABLE = importlib.util.find_spec("prophet") is not None
if PROPHET_AVAILABLE:
    from prophet import Prophet

    for noisy in ("cmdstanpy", "prophet"):
        logging.getLogger(noisy).setLevel(logging.ERROR)

series = pd.read_parquet(nb_config.CDC_TEMP_PATH)["Brandenburg/Berlin"].asfreq("MS")
TEST_MONTHS, SEASON_LENGTH = 24, 12
train, test = series.iloc[:-TEST_MONTHS], series.iloc[-TEST_MONTHS:]


def mean_absolute_error(actual, forecast):
    return float(np.mean(np.abs(np.asarray(actual) - np.asarray(forecast))))


print(f"Prophet available: {PROPHET_AVAILABLE}")

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-1">Exercise 1</h3>
</div>

> `ThetaModel` takes a `deseasonalize` argument, on by default. Refit with `deseasonalize=False` and compare. How much of Theta's performance on this series comes from the seasonal adjustment rather than from the theta lines themselves?

In [ ]:
results = {}
for deseasonalize in (True, False):
    fitted = ThetaModel(train, period=SEASON_LENGTH, deseasonalize=deseasonalize).fit()
    results[deseasonalize] = fitted.forecast(TEST_MONTHS)
    print(f"deseasonalize={deseasonalize!s:<5}  MAE "
          f"{mean_absolute_error(test, results[deseasonalize]):.2f} °C")

print()
print("For comparison, from Notebooks A05 and B01:")
print("  Seasonal naive  1.74     Mean  6.04     Holt-Winters  1.22")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.5))

ax.plot(train["2021":], color="steelblue", linewidth=1.2, label="Train")
ax.plot(test, color="black", linewidth=1.8, label="Actual")
ax.plot(results[True], color="seagreen", linewidth=1.5, linestyle="--",
        label="Theta, deseasonalised")
ax.plot(results[False], color="crimson", linewidth=1.5, linestyle="--",
        label="Theta, raw series")

ax.axvline(test.index[0], color="gray", linestyle="--", linewidth=1.0)
ax.set_title("Theta with and without the seasonal adjustment", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Temperature (°C)")
ax.legend(loc="upper left")
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

**Essentially all of it.** With the seasonal adjustment Theta scores 1.25 °C; without it, 5.67 °C — worse
than the seasonal naive baseline, and squarely among the flat forecasts of Notebook A05, whose Mean scored
6.04.

The plot shows why. Deseasonalised Theta produces a forecast with a seasonal shape; raw Theta produces
something close to a straight line. The theta lines themselves model **curvature and trend**, and this
series has almost no trend and no curvature worth speaking of — what it has is a large, stable annual
cycle, which the theta decomposition does not address at all.

That is a useful thing to know about a method with a reputation for winning competitions. Theta's M3
result was not magic: it was a simple, robust trend model **wrapped in a classical seasonal adjustment**,
applied to a competition whose series were mostly seasonal. On this data the wrapper is doing the work.

It is also a reason not to over-read the comparison in the notebook's own summary table. Theta at 1.25 °C
sits close to Holt-Winters at 1.22 and ahead of SARIMA at 1.31, which makes it look like a competitive
alternative — and it is. But all three are, underneath, doing the same thing: estimating a stable seasonal
profile and adding a small amount of level and trend on top. They agree because the series leaves little
else to disagree about.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-2">Exercise 2</h3>
</div>

> Add the previous day's load at the same hour as an exogenous column alongside the Fourier terms, and refit. Does giving the model the level directly close the gap to the naive forecast?

In [ ]:
ops = pd.read_parquet(nb_config.OPS_15M_PATH)

load = (
    ops[(ops["country"] == "AT") & (ops["measure"] == "actual_entsoe_transparency")]["value"]
    .tz_convert(None)
    .resample("h").mean()
    .dropna()
    .asfreq("h")
)

HORIZON = 24
TRAINING_WEEKS = 8
lagged_load = load.shift(24)      # the same hour, one day earlier


def harmonic_terms(index, horizon):
    process = DeterministicProcess(
        index, constant=True,
        additional_terms=[Fourier(24, order=4), Fourier(168, order=3)],
    )
    return process.in_sample(), process.out_of_sample(horizon)


def evaluate_at(origin):
    """Fit both variants at one origin and score them against the naive forecast."""
    history = load.loc[origin - pd.Timedelta(weeks=TRAINING_WEEKS):origin - pd.Timedelta(hours=1)]
    actual = load.loc[origin:origin + pd.Timedelta(hours=HORIZON - 1)]

    exog_train, exog_future = harmonic_terms(history.index, HORIZON)

    plain = SARIMAX(history, exog=exog_train, order=(2, 0, 1),
                    enforce_stationarity=False).fit(disp=False)

    # The same design matrix plus yesterday's load at the matching hour
    with_lag_train = exog_train.assign(lag_24=lagged_load.loc[exog_train.index].values)
    with_lag_future = exog_future.assign(lag_24=lagged_load.loc[exog_future.index].values)
    usable = ~with_lag_train.isna().any(axis=1)

    with_lag = SARIMAX(history[usable.values], exog=with_lag_train[usable],
                       order=(2, 0, 1), enforce_stationarity=False).fit(disp=False)

    return {
        "origin": origin.date(),
        "DHR": mean_absolute_error(actual, plain.forecast(HORIZON, exog=exog_future)),
        "DHR + lag 24": mean_absolute_error(
            actual, with_lag.forecast(HORIZON, exog=with_lag_future)
        ),
        "Naive": mean_absolute_error(actual, history.iloc[-24:].values),
    }


origins = pd.date_range("2018-03-15", periods=6, freq="21D")
by_origin = pd.DataFrame([evaluate_at(origin) for origin in origins]).set_index("origin")

by_origin.round(0)

In [ ]:
print(by_origin.mean().round(1).to_string())

**Yes — almost exactly.** Adding one column takes dynamic harmonic regression from 436 MW to **204 MW**,
and the naive forecast it was losing to scores **203 MW**. The gap does not narrow; it closes.

That is a satisfying confirmation of the diagnosis in section 4 of the notebook. The argument there was
that DHR's Fourier terms describe an *average* day, carrying no information about the current level,
while the naive forecast consists of nothing but the current level. If that diagnosis was right, handing
the model the level should recover the difference — and it does, to within 2 MW.

Two things follow.

**A model losing to a naive baseline is usually missing something specific.** It is tempting to conclude
that the method is unsuited to the problem, and sometimes that is true. Here the harmonic model was fine
and simply blind to one input, which one column fixed. Working out *what* a baseline knows that your model
does not is more productive than replacing the model.

**And this is why production load forecasting looks the way it does.** Nobody deploys bare Fourier terms.
The deterministic seasonal profile is one component, combined with recent levels and a weather forecast,
which is roughly the architecture this exercise has just reconstructed from two pieces.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-3">Exercise 3</h3>
</div>

> Prophet's holiday support is its strongest feature, and this temperature series cannot show it. Fit Prophet to the Rossmann daily sales from Notebook A04, passing the `StateHoliday` dates via the `holidays` argument. How much does declaring the holidays improve the forecast?

In [ ]:
sales = pd.read_csv(nb_config.ROSSMANN_TRAIN_PATH, parse_dates=["Date"], low_memory=False)
store = sales[sales["Store"] == 1].set_index("Date").sort_index().asfreq("D")
daily_sales = store["Sales"].astype(float)

holiday_dates = store.index[
    (store["StateHoliday"].astype(str) != "0") & store["StateHoliday"].notna()
]

holidays = pd.DataFrame({
    "holiday": "state_holiday",
    "ds": holiday_dates,
    "lower_window": 0,
    "upper_window": 0,
})

print(f"{len(holiday_dates)} state holidays across {len(daily_sales)} days "
      f"({len(holiday_dates) / len(daily_sales):.1%})")


def fit_prophet(train_frame, future_index, use_holidays):
    model = Prophet(
        weekly_seasonality=True, yearly_seasonality=True, daily_seasonality=False,
        holidays=holidays if use_holidays else None,
    )
    model.fit(train_frame)
    return model.predict(pd.DataFrame({"ds": future_index}))["yhat"].values

In [ ]:
if PROPHET_AVAILABLE:
    SHORT_HORIZON = 28
    train_frame = pd.DataFrame({
        "ds": daily_sales.index[:-SHORT_HORIZON], "y": daily_sales.values[:-SHORT_HORIZON]
    })
    short_test = daily_sales.iloc[-SHORT_HORIZON:]

    for use_holidays in (False, True):
        predicted = fit_prophet(train_frame, short_test.index, use_holidays)
        print(f"holidays={use_holidays!s:<5}  MAE {mean_absolute_error(short_test, predicted):7.1f}")

    print(f"\nHolidays inside this {SHORT_HORIZON}-day test window: "
          f"{int(short_test.index.isin(holiday_dates).sum())}")

**Declaring the holidays makes the forecast very slightly worse**, 540 against 536 — and the last line
explains why. There is **not one holiday in the test window**.

This is the trap the exercise sets, and it is worth failing once. Prophet has been told about 27 holidays
in the training data and has fitted an effect for them, which costs a few degrees of freedom. Over 28 days
containing no holidays that effect can only add noise, and the measured difference is noise.

The feature cannot help on days it does not apply to. To measure it, the test period has to contain the
event.

In [ ]:
if PROPHET_AVAILABLE:
    LONG_HORIZON = 180
    long_train = pd.DataFrame({
        "ds": daily_sales.index[:-LONG_HORIZON], "y": daily_sales.values[:-LONG_HORIZON]
    })
    long_test = daily_sales.iloc[-LONG_HORIZON:]
    on_holiday = long_test.index.isin(holiday_dates)

    rows = []
    for use_holidays in (False, True):
        predicted = fit_prophet(long_train, long_test.index, use_holidays)
        rows.append({
            "holidays declared": use_holidays,
            "overall": mean_absolute_error(long_test, predicted),
            "on holidays": mean_absolute_error(long_test[on_holiday], predicted[on_holiday]),
            "other days": mean_absolute_error(long_test[~on_holiday], predicted[~on_holiday]),
        })

    print(f"{LONG_HORIZON}-day test window, containing {int(on_holiday.sum())} holidays\n")
    display(pd.DataFrame(rows).set_index("holidays declared").round(1))

Now the feature can show what it does, and the answer is dramatic.

**On holiday days the error falls from 4,407 to 240 — a reduction of 95%.** Without the holiday
information Prophet forecasts an ordinary day and the shop is shut, so it is wrong by roughly a full day's
sales. With it, those days are predicted almost exactly.

Two details in that table are worth more than the headline.

**The overall improvement is only 14%**, from 722 to 617. Six days out of 180 cannot move an average very
far, however badly they were forecast. A feature that fixes a catastrophic error on 3% of days looks
modest in aggregate, and a single summary number is exactly the wrong instrument for finding it.

**And ordinary days got slightly worse**, 595 to 630. The holiday component has to be estimated, it
absorbs some variation that is not really holiday-related, and the rest of the model pays a little for it.
That is the normal cost of adding a feature, and it is worth the trade here by a wide margin.

The general lesson is about evaluation rather than about Prophet. **Break the error down by the condition
the feature addresses.** Had we stopped at the first test, the honest conclusion would have been that
declaring holidays does not help — which is true of that window and false in general.

---

Back to [Notebook B03](../notebooks/B03_Advanced_statistical_models.ipynb), or on to
[Notebook B04](../notebooks/B04_Probabilistic_forecasting.ipynb).